# SharePoint Verification
Confirm SharePoint connection has been configured and docs can be ingested. 

---


## Unity Catalog Connection
Create a Unity Catalog connection to store your SharePoint credentials. 
https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/sharepoint-connection
- **OAuth U2M: Databricks-managed** - The account used to authenticate is shared with all users.  Use a service account!
- **OAuth U2M: Custom-managed** - organization requires control over the OAuth application
- **OAuth M2M** - Use M2M for fully automated production pipelines that run without user interaction.

Create the connection manually through the Catalog Explorer UI, then update the connection_name in the Variable definitions cell, below.

In [0]:
dbutils.widgets.text("catalog", "dennis_schultz", "Catalog Name")
dbutils.widgets.text("schema", "vantage_workshop", "Schema Name")
dbutils.widgets.text("bronze_table", "01_bronze", "Bronze Table Name")
dbutils.widgets.text("connector_name", "vantage_demo_sharepoint_connection", "SharePoint Connection Name")

# Unity Catalog
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
bronze_table = dbutils.widgets.get("bronze_table")
connection_name = dbutils.widgets.get("connector_name")

# SharePoint
sharepoint_site_url = "https://databricksinc.sharepoint.com/sites/VantageDemo/Shared%20Documents/Forms/AllItems.aspx"

excel_sharepoint_file_url = "https://databricksinc.sharepoint.com/sites/VantageDemo/Shared%20Documents/Sample-Superstore.xls"
excel_bronze_table_name = f"{bronze_table}_excel"
excel_dataAddress = "Orders"

In [0]:
# Set the default catalog and schema for this session
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
if spark.catalog.tableExists(bronze_table):
    spark.sql(f"DELETE FROM {bronze_table}")


## Unstructured

### PDF documents

In [0]:
from pyspark.sql.functions import lit

# Read all PDF files from SharePoint as binary files

# This is a batch read. For automatic incremental ingestion, view the cells at the bottom of the notebook to see how to use this connector in Lakeflow Spark Declarative Pipelines
pdf_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", connection_name) # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pdf")  # Only ingest PDF files
    .load(sharepoint_site_url)
    .select("*", "_metadata")
    .withColumn("source", lit("SharePoint"))
)

# Display the DataFrame to see the PDF files
display(pdf_df.limit(1))

# Save the PDF files to a Delta table for persistent storage
pdf_df.write \
    .mode("append") \
    .saveAsTable(bronze_table)


### PowerPoint documents

In [0]:
from pyspark.sql.functions import lit

# Read all Powerpoint files from SharePoint as binary files

ppt_df = (spark.read
    .format("binaryFile") # Use this format for unstructured data
    .option("databricks.connection", connection_name) # User provides the name of their connection
    .option("recursiveFileLookup", True)
    .option("pathGlobFilter", "*.pptx")  # Only ingest PowerPoint files
    .load(sharepoint_site_url)
    .select("*", "_metadata")
    .withColumn("source", lit("SharePoint"))
)

# Save the Powerpoint files to a Delta table for persistent storage
ppt_df.write \
    .mode("append") \
    .saveAsTable(bronze_table)

# Display the DataFrame with only the first 100 bytes of file content
display(
    ppt_df.selectExpr(
        "path",
        "modificationTime",
        "length",
        "substring(content, 1, 100) as content_first_100_bytes",
        "_metadata",
        "source"
    )
)

## Show current Bronze table

In [0]:
display(
    spark.read.table(bronze_table)
        .selectExpr("path",
            "modificationTime",
            "length",
            "substring(content, 1, 100) as content_first_100_bytes",
            "_metadata",
            "source")
)

## Structured


### Sync Individual Excel Files to Delta Tables

This section demonstrates how to:
1. Read a specific Excel file from SharePoint using Python `spark.read()` or SQL `read_files()`

Excel files support options like `headerRows` and `dataAddress` to specify which sheet and range to read.

In [0]:
# Read a specific Excel file from SharePoint
excel_df = (spark.read
    .format("excel")
    .option("databricks.connection", connection_name)
    .option("headerRows", 1)  # First row contains headers
    .option("inferSchema", True)  # Automatically infer column types
    .option("dataAddress", excel_dataAddress) # Specify the tab and cell range, if required
    .load(excel_sharepoint_file_url)
    .withColumn("source", lit("SharePoint"))
)

# Save the Excel data to a Delta table
(excel_df.write \
    .option("delta.columnMapping.mode", "name")  # Column mapping enables the use of non-standard column names
    .mode("overwrite") 
    .saveAsTable(excel_bronze_table_name)
)

# Display the Excel data
display(excel_df)